In [1]:
# LogEc – Automated Download of Working Paper Statistics

# This notebook downloads and parses statistics tables from LogEc for the
# Working Paper Series of the School of Government and Public Transformation
# (Tecnológico de Monterrey).
#
# It retrieves the Top-25 tables for:
# - Last 3 months
# - Last 12 months
# - Total downloads
#
# And outputs three pandas DataFrames and CSV files, including authors.

# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

import sys
from dataclasses import dataclass
from typing import Dict, List

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [2]:
# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

BASE_URL = "https://logec.repec.org/scripts/seritemstat.pf"

COMMON_PARAMS = {
    "h": "repec:gnt:wpaper",
    "topnum": 30,
}

SORT_OPTIONS: Dict[str, str] = {
    "last_3_months": "3d",
    "last_12_months": "12d",
    "total": "td",
}

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/119.0.0.0 Safari/537.36"
)


In [3]:
# ------------------------------------------------------------
# 3. Data structure
# ------------------------------------------------------------

@dataclass
class PaperStat:
    rank: str
    title: str
    authors: str
    fd_2025_10: str
    fd_3_months: str
    fd_12_months: str
    fd_total: str
    av_2025_10: str
    av_3_months: str
    av_12_months: str
    av_total: str

    def to_dict(self) -> Dict[str, str]:
        return {
            "Rank": self.rank,
            "Working Paper": self.title,
            "Authors": self.authors,
            "FD_2025_10": self.fd_2025_10,
            "FD_3_months": self.fd_3_months,
            "FD_12_months": self.fd_12_months,
            "FD_Total": self.fd_total,
            "AV_2025_10": self.av_2025_10,
            "AV_3_months": self.av_3_months,
            "AV_12_months": self.av_12_months,
            "AV_Total": self.av_total,
        }

In [4]:
# ------------------------------------------------------------
# 4. Function to fetch and parse a table
# ------------------------------------------------------------

def fetch_table(sortby: str) -> pd.DataFrame:
    params = COMMON_PARAMS.copy()
    params["sortby"] = sortby

    headers = {"User-Agent": USER_AGENT}

    response = requests.get(BASE_URL, params=params, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table", class_="stats")
    if table is None:
        raise RuntimeError("Statistics table not found")

    stats: List[PaperStat] = []

    for row in table.find_all("tr"):
        rank_cell = row.find("td", class_="statrank")
        if rank_cell is None:
            continue

        cells = row.find_all("td")
        if len(cells) < 10:
            continue

        rank = cells[0].get_text(strip=True)

        title_link = cells[1].find("a")
        title = title_link.get_text(strip=True) if title_link else ""

        author_tags = cells[1].find_all("i")
        authors = ", ".join(a.get_text(strip=True) for a in author_tags)

        numbers = [c.get_text(strip=True).replace("\xa0", " ") for c in cells[2:]]
        numbers = numbers[:8]

        stats.append(
            PaperStat(
                rank=rank,
                title=title,
                authors=authors,
                fd_2025_10=numbers[0],
                fd_3_months=numbers[1],
                fd_12_months=numbers[2],
                fd_total=numbers[3],
                av_2025_10=numbers[4],
                av_3_months=numbers[5],
                av_12_months=numbers[6],
                av_total=numbers[7],
            )
        )

    return pd.DataFrame([p.to_dict() for p in stats])

In [5]:
# ------------------------------------------------------------
# 5. Run downloads
# ------------------------------------------------------------

frames = {}

for name, sortby in SORT_OPTIONS.items():
    df = fetch_table(sortby)
    frames[name] = df
    display(df)
    df.to_excel(f"data/gnt_{name}.xlsx", index=False)

,Rank,Working Paper,Authors,FD_2025_10,FD_3_months,FD_12_months,FD_Total,AV_2025_10,AV_3_months,AV_12_months,AV_Total
0,1,"Chinese Investment in Mexico: Trade Wars, Near...","Agustina Giraudy, Ernesto Stein, Francisco Urd...",2,21,21,21,7,40,40,40
1,2,Beneficiation vs. Knowledge-based: Dead ends a...,"Sebastián Bustos, Miguel Santos",0,11,11,11,4,15,15,15
2,3,Proximity as a Substitute of Contract Enforcem...,"Luis Espinoza, Jose Morales-Arilla",0,10,10,10,3,11,11,11
3,4,A discrete choice model for labor informality ...,"Hector Villarreal, Diego Vázquez-Pimentel",3,5,29,29,8,17,44,44
4,5,The Influence of TikTok on Political Campaigns...,"Fernanda Sobrino, Alejandro Díaz Domínguez",1,4,25,25,3,12,62,62
5,5,La Complejidad Económica de El Salvador: Usand...,"Fernando Gómez-Zaldívar, Hermilo Cortés, Vícto...",0,4,5,5,3,11,17,17
6,7,Estado de la implementación de políticas de nu...,"Paola Abril Campos Rivera, Berenice Alfaro Pon...",2,3,4,4,5,11,20,20
7,8,Economic Complexity and Regional Manufacturing...,"Manuel Gómez-Zaldívar, Fernando Gómez-Zaldívar",0,2,9,9,3,13,20,20
8,8,Adequate and Affordable Housing for 2040 Metro...,"Steven W. Popper, Jose Antonio Torre, Eduardo ...",2,2,2,2,6,6,6,6
9,8,Codebook: Subnational Politics Project (SPP),"Agustina Giraudy, Francisco Urdinez, Guadalupe...",2,2,2,2,10,10,10,10


,Rank,Working Paper,Authors,FD_2025_10,FD_3_months,FD_12_months,FD_Total,AV_2025_10,AV_3_months,AV_12_months,AV_Total
0,1,A discrete choice model for labor informality ...,"Hector Villarreal, Diego Vázquez-Pimentel",3,5,29,29,8,17,44,44
1,2,The Influence of TikTok on Political Campaigns...,"Fernanda Sobrino, Alejandro Díaz Domínguez",1,4,25,25,3,12,62,62
2,3,On the Dynamics of Mental Health,"Diego Ascarza-Mendoza, Christian Velasquez",0,0,23,23,5,10,38,38
3,4,"Chinese Investment in Mexico: Trade Wars, Near...","Agustina Giraudy, Ernesto Stein, Francisco Urd...",2,21,21,21,7,40,40,40
4,5,Beneficiation vs. Knowledge-based: Dead ends a...,"Sebastián Bustos, Miguel Santos",0,11,11,11,4,15,15,15
5,6,The Impact of Artificial Intelligent Tools on ...,"Edmundo Molina-Perez, Pedro Cortes, Isaac Moli...",0,0,10,10,2,3,39,39
6,6,Proximity as a Substitute of Contract Enforcem...,"Luis Espinoza, Jose Morales-Arilla",0,10,10,10,3,11,11,11
7,8,Economic Complexity and Regional Manufacturing...,"Manuel Gómez-Zaldívar, Fernando Gómez-Zaldívar",0,2,9,9,3,13,20,20
8,9,La Complejidad Económica de El Salvador: Usand...,"Fernando Gómez-Zaldívar, Hermilo Cortés, Vícto...",0,4,5,5,3,11,17,17
9,10,Estado de la implementación de políticas de nu...,"Paola Abril Campos Rivera, Berenice Alfaro Pon...",2,3,4,4,5,11,20,20


,Rank,Working Paper,Authors,FD_2025_10,FD_3_months,FD_12_months,FD_Total,AV_2025_10,AV_3_months,AV_12_months,AV_Total
0,1,A discrete choice model for labor informality ...,"Hector Villarreal, Diego Vázquez-Pimentel",3,5,29,29,8,17,44,44
1,2,The Influence of TikTok on Political Campaigns...,"Fernanda Sobrino, Alejandro Díaz Domínguez",1,4,25,25,3,12,62,62
2,3,On the Dynamics of Mental Health,"Diego Ascarza-Mendoza, Christian Velasquez",0,0,23,23,5,10,38,38
3,4,"Chinese Investment in Mexico: Trade Wars, Near...","Agustina Giraudy, Ernesto Stein, Francisco Urd...",2,21,21,21,7,40,40,40
4,5,Beneficiation vs. Knowledge-based: Dead ends a...,"Sebastián Bustos, Miguel Santos",0,11,11,11,4,15,15,15
5,6,The Impact of Artificial Intelligent Tools on ...,"Edmundo Molina-Perez, Pedro Cortes, Isaac Moli...",0,0,10,10,2,3,39,39
6,6,Proximity as a Substitute of Contract Enforcem...,"Luis Espinoza, Jose Morales-Arilla",0,10,10,10,3,11,11,11
7,8,Economic Complexity and Regional Manufacturing...,"Manuel Gómez-Zaldívar, Fernando Gómez-Zaldívar",0,2,9,9,3,13,20,20
8,9,La Complejidad Económica de El Salvador: Usand...,"Fernando Gómez-Zaldívar, Hermilo Cortés, Vícto...",0,4,5,5,3,11,17,17
9,10,Estado de la implementación de políticas de nu...,"Paola Abril Campos Rivera, Berenice Alfaro Pon...",2,3,4,4,5,11,20,20
